# Training an Emulator and Fitting some Data

## Environment setup

`pip install torch torchvision torchaudio numpy matplotlib tqdm jupyter ultranest scikit-learn astropy numba`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn

import numba
from tqdm.notebook import tqdm

torch.random.manual_seed(42);

# Our Mock-Simulation that we would like to emulate

Create a rough simulation of a supernova spectrum

In [ ]:
from astropy import units as u, constants as const

const_h = const.h.cgs.value
const_c = const.c.cgs.value
const_kb = const.k_B.cgs.value


@numba.njit
def P_cygni(grid, nu, vel, amp): # Approximate P-Cygni profile
    
    wave = const_c / nu
    wave_grid = const_c/grid
    absorption = np.ones(len(grid))
    wave_m = wave*vel/const_c

    absorption[np.where((wave_grid < wave) & (wave_grid > wave-wave_m))] = 0.0
    absorption /= absorption.sum()
    absorption -= 0.5/len(absorption)
    
    emission = (np.exp(-((wave - wave_grid)**2)/(2*(wave_m/3)**2)))
    emission /= emission.sum()

    profile = emission + absorption

    return profile

@numba.njit
def create_spectrum(nu, T, amp1, amp2, amp3, vel1, vel2, vel3): # A simulated spectrum
    
    BB =  2 * const_h * nu**3 / const_c**2 * 1 / (np.exp((nu * const_h) / (const_kb * T)) - 1)

    BB += P_cygni(nu, 2e14, vel1, 1e-1)*amp1
    BB += P_cygni(nu, 1e14, vel2, 1e-1)*amp2
    BB += P_cygni(nu, 5e13, vel3, 1e-1)*amp3
    
    return BB

@numba.njit(parallel=True)
def gen_spectra(parameters): # Generate spectra from a grid of input parameters
    
    spectra = np.empty((len(parameters), len(hz_grid)))
    for i in numba.prange(len(parameters)):
        T, amp1, amp2, amp3, vel1, vel2, vel3 = parameters[i]
        spectra[i] = create_spectrum(hz_grid, T, amp1, amp2, amp3, vel1, vel2, vel3)
    return spectra

In [ ]:
hz_grid = np.logspace(12, 15, 1000) # Frequency grid
spec = create_spectrum(hz_grid, 5000.0, 5e-5, 5.5e-5, 2e-4, 1e9, 2e9, 5e9)

plt.figure(dpi=150)

plt.plot(const_c/hz_grid, spec) # Plot in wavelength space

plt.xscale('log')
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')

# Generate a Training Set

In [ ]:
N = 50000 # Number of examples
hz_grid = np.logspace(12, 15, 1000)
temperatures = np.random.random(N) * (10000 - 2000) + 2000 # Random grid of temperatures
amp1s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5) # Random grids of amplitudes
amp2s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5) 
amp3s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5)

vel1s = 10**(np.random.random(N) * (10 - 9) + 9) # Random grid of Velocities
vel2s = 10**(np.random.random(N) * (10 - 9) + 9)
vel3s = 10**(np.random.random(N) * (10 - 9) + 9)

params = np.vstack((temperatures, amp1s, amp2s, amp3s, vel1s, vel2s, vel3s)).T

spectra = gen_spectra(params) # Generate N spectra

X = params
Y = spectra

In [ ]:
plt.figure(dpi=150)

plt.plot(const_c/hz_grid, spectra[0])
plt.plot(const_c/hz_grid, spectra[1])
plt.plot(const_c/hz_grid, spectra[2])

plt.xscale('log')
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')

In [ ]:
plt.figure()
plt.hist(X[:, 1], histtype='step')
plt.xlabel("Amplitudes")

plt.figure()
plt.hist(Y.ravel(), histtype='step')
plt.xlabel("Spectrum Flux")

# Preprocessing

  Neural Networks tend to work better when input points are around the same scale close to 1

In [ ]:
X_nn = torch.from_numpy(X.copy()).to(torch.float32) # The model runs faster with 32-point precision
X_nn[:, 1:] = torch.log10(X_nn[:, 1:]) # The distribution for the amplitudes and velocities is logarithmic

X_min = X_nn.min(axis=0).values
X_max = X_nn.max(axis=0).values

X_nn = (X_nn - X_min) / (X_max - X_min) # Scale the parameters to be from 0 to 1

Y_nn = torch.from_numpy(Y.copy()).to(torch.float32)
Y_nn = torch.log10(Y_nn) # The spectra themselves tend to be log-distributed

Y_min = Y_nn.min(axis=0).values
Y_max = Y_nn.max(axis=0).values

Y_nn = (Y_nn - Y_min) / (Y_max - Y_min) # Scale the spectra from 0 to 1

In [ ]:
plt.figure()
plt.hist(X_nn[:, 1], histtype='step')
plt.xlabel("Amplitudes")

plt.figure()
plt.hist(Y_nn.ravel(), histtype='step')
plt.xlabel("Spectrum Flux")

plt.figure()
plt.plot(const_c/hz_grid, Y_nn.numpy()[0])
plt.xlabel('Wavelength (cm)')
plt.ylabel('Scaled Flux')
plt.xscale('log')

# Train and Validation Data

Some data should be held out to validate the quality of the emulator

In [ ]:
# Split the data into training and validation sets
from sklearn.model_selection import train_test_split

(X_train, 
 X_val,
 Y_train,
 Y_val) = train_test_split(X_nn, Y_nn, train_size=0.7, random_state=42, shuffle=True)

# Creating the model

In [ ]:
class Emulator(nn.Module): # All models are subclasses of nn.Module

    def __init__(self, n_input, n_output, n_hidden):

        super().__init__()

        self.n_input = n_input
        self.n_hidden = n_hidden
        self.n_output = n_output

        self.input_layer = nn.Sequential(
            nn.Linear(n_input, n_hidden), # Matrix Multiplication
            nn.Softplus() # Activation Function
        )
        self.hidden_layer = nn.Sequential(
            nn.Linear(n_hidden, n_hidden),
            nn.Softplus(),
        )
        self.output_layer = nn.Linear(n_hidden, n_output)

    def forward(self, x): # Function that takes in an input and returns a prediction
        # x is (Batch x n_input)
        
        x = self.input_layer(x) # (Batch x n_input) -> (Batch x n_hidden)
        x = self.hidden_layer(x) # (Batch x n_hidden) -> (Batch x n_hidden)
        x = self.output_layer(x) # (Batch x n_hidden) -> (Batch x n_output)
        
        return x
        

# Test out the model

In [ ]:
emulator = Emulator(
    n_input=X_nn.shape[1],
    n_hidden=64,
    n_output=Y_nn.shape[1]
)
print(emulator)

In [ ]:
Y_pred = emulator(X_train[:1])
Y_pred = Y_pred
Y_pred = 10**(Y_pred * (Y_max - Y_min) + Y_min)
Y_pred = Y_pred.detach().numpy()

plt.plot(const_c/hz_grid, Y_pred[0])
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')
plt.xscale('log')

# The Training Loop - The Optimizer

In [ ]:
learning_rate = 0.001

optimizer = torch.optim.Adam(
    emulator.parameters(), 
    lr=learning_rate
)

# The Training Loop - The Loss Function

In [ ]:
def mean_square_error(Y_true, Y_pred):

    return torch.mean((Y_true - Y_pred) ** 2)

# The Training Loop - The Training Loop

In [ ]:
def train_model(model, optimizer, loss_function, X_train, Y_train, batch_size=100, epoch=0):

    # Split the data into batches (Usually for memory constraints)
    num_batches = len(X_nn) // batch_size
    batches_X = torch.chunk(X_train, num_batches)
    batches_Y = torch.chunk(Y_train, num_batches)

    loss_curve = []

    for step in (pbar:=tqdm(range(num_batches), leave=False, desc=f'Training epoch {epoch}')):
        model.zero_grad() # Reset the model gradients
        
        X_input = batches_X[step] # Get the input data for the current training step
        Y_pred = model(X_input) # Evaluate the model Prediction
        Y_true = batches_Y[step] # Get the true value

        loss = loss_function(Y_true, Y_pred) # Compute the loss
        loss.backward() # Compute the gradients on the loss

        optimizer.step() # Update the weights

        loss_curve.append(loss.item()) # Save the value of the loss at each step
        pbar.set_postfix(loss=f"{loss.item():2.2e}")
    return loss_curve


# Training the Model

In [ ]:
loss_curve = train_model(
    emulator,
    optimizer,
    mean_square_error,
    X_train,
    Y_train,
    batch_size=100
)

In [ ]:
plt.plot(loss_curve)
plt.xlabel('Epoch')
plt.ylabel('Loss')

# Validating the model

In [ ]:
def show_predictions(model, loss_function, X_val, Y_val):
    with torch.inference_mode(): # Don't compute gradients
        Y_pred = model(X_val) # Evaluate the model
    Y_pred = Y_pred * (Y_max - Y_min) + Y_min # Invert the preprocessing
    Y_pred = 10**Y_pred.detach().numpy() # Turn back to numpy array
    
    loss = loss_function(Y_val, Y_pred)
    print("Validation Loss:", loss)
    
    Y_true = Y_val * (Y_max - Y_min) + Y_min
    Y_true = 10**Y_true.detach().numpy()
    
    for i in range(3):
        plt.figure()
        plt.plot(const_c/hz_grid, Y_true[i], label='True Spectrum')
        plt.plot(const_c/hz_grid, Y_pred[i], label='Predicted Spectrum', ls='--')
        plt.xlabel('Wavelength (cm)')
        plt.ylabel('Flux')
        plt.legend()
        plt.xscale('log')

show_predictions(emulator, mean_square_error, X_val, Y_val)



# Improve Training Loop to Track the Validation Loss

In [ ]:
def evaluate_validation_loss(model, loss_function, X_val, Y_val, batch_size=100, epoch=0):

     # Split the data into batches
    num_batches = len(X_nn) // batch_size
    batches_X = torch.chunk(X_val, num_batches)
    batches_Y = torch.chunk(Y_val, num_batches)
    
    total_loss = 0 # Track the total loss

    with torch.inference_mode(): # Turn off gradient computation
        for step in tqdm(range(num_batches), leave=False, desc=f'Validation epoch {epoch}'):
            X_input = batches_X[step] # Get the input data for the current training step
            Y_pred = model(X_input) # Evaluate the model Prediction
            Y_true = batches_Y[step] # Get the true value
    
            loss = loss_function(Y_true, Y_pred) # Compute the loss
            total_loss += loss.item()
            
    average_loss = total_loss / num_batches
    return average_loss

In [ ]:
val_loss = evaluate_validation_loss(emulator, mean_square_error, X_val, Y_val)
print("Validation Loss:", val_loss)

# Create a Training Loop that Perioidically Tracks Validation Loss

In [ ]:
def train_model_with_loss_tracking(model, optimizer, loss_function, X_train, Y_train, X_val, Y_val, batch_size=100, num_epochs=50, show_prediction_plots=True):

    train_loss_curve = []
    val_loss_curve = []
    for epoch in (pbar:=tqdm(range(num_epochs))):
        train_loss = train_model(model, optimizer, loss_function, X_train, Y_train, batch_size=batch_size, epoch=epoch)
        val_loss = evaluate_validation_loss(model, loss_function, X_val, Y_val, batch_size=batch_size, epoch=epoch)

        train_loss_curve.append(np.mean(train_loss)) # Average training loss over this epoch
        val_loss_curve.append(val_loss)

        pbar.set_postfix(train_loss=f"{train_loss_curve[-1]:2.2e}", validation_loss=f"{val_loss_curve[-1]:2.2e}")

    plt.figure()
    plt.plot(train_loss_curve, label='Training')
    plt.plot(val_loss_curve, label='Validation')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.legend()

    if show_prediction_plots:
        show_predictions(model, loss_function, X_val, Y_val)

In [ ]:
train_model_with_loss_tracking(
    emulator, 
    optimizer, 
    mean_square_error,
    X_train, 
    Y_train, 
    X_val, 
    Y_val, 
    batch_size=100,
    num_epochs=50
)

# Task For You

Train the model until learning has plateaued 

How many total epochs did it take?

How does batch size affect training?

How does learning rate affect training?

What kind of plots or metrics can we show to capture the model's performance over all validation samples?

# Fitting Data

In [ ]:
# Generate some realistic looking data with noise
np.random.seed(47)

temperature = np.random.random() * (10000 - 2000) + 2000 # Random grid of temperatures
amp1 = 10**(np.random.random() * (-4 - -5.5) + -5.5) # Random grids of amplitudes
amp2 = 10**(np.random.random() * (-4 - -5.5) + -5.5) 
amp3 = 10**(np.random.random() * (-4 - -5.5) + -5.5)

vel1 = 10**(np.random.random() * (10 - 9) + 9) # Random grid of Velocities
vel2 = 10**(np.random.random() * (10 - 9) + 9)
vel3 = 10**(np.random.random() * (10 - 9) + 9)

sample_params = np.array([[temperature, amp1, amp2, amp3, vel1, vel2, vel3]])

sample_spectrum = gen_spectra(sample_params)[0]
sample_spectrum_int = sample_spectrum.sum()
sample_spectrum_norm = sample_spectrum / sample_spectrum_int

photons = np.random.random((len(hz_grid), 50000)) # Simulated Photons, lower number -> More uncertainty
counts = np.array([np.sum(photons[i] < sample_spectrum_norm[i]) for i in range(len(hz_grid))])
counts_int = counts.sum()

sample_spectrum = counts / counts_int * sample_spectrum_int
sample_spectrum_err = np.sqrt(counts) / counts_int * sample_spectrum_int
sample_spectrum_err[sample_spectrum_err == 0] = 1 / counts_int * sample_spectrum_int

plt.figure(dpi=150)
plt.errorbar(const_c/hz_grid, sample_spectrum, yerr=sample_spectrum_err)
plt.xscale('log')
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')

param_names = ['Temperature', 'Amplitude 1', 'Amplitude 2', 'Amplitude 3', 'Velocity 1', 'Velocity 2', 'Velocity 3']

for key, value in zip(param_names, sample_params[0]):
    print(f'{key:<15s}{value:>2.2e}')

# Performing Inference

In [ ]:
import ultranest

def prior_transform(cube): # Set up a fuction that converts a grid of uniform-random numbers into model parameters
    
    # Transform random uniforms into our parameters
    cube = cube.copy()
    
    cube[:, 0] = cube[:, 0] * (10000 - 2000) + 2000 # Temperature
     
    cube[:, 1] = 10**(cube[:, 1] * (-4 - -5.5) + -5.5) # Amplitudes
    cube[:, 2] = 10**(cube[:, 2] * (-4 - -5.5) + -5.5)
    cube[:, 3] = 10**(cube[:, 3] * (-4 - -5.5) + -5.5)
    
    cube[:, 4] = 10**(cube[:, 4] * (10 - 9) + 9) # Velocities
    cube[:, 5] = 10**(cube[:, 5] * (10 - 9) + 9)
    cube[:, 6] = 10**(cube[:, 6] * (10 - 9) + 9)
    
    return cube
    
def likelihood(parameters): # Compute the likelihood of data | parameters
    
    # Preprocess the data
    parameters_nn = torch.from_numpy(parameters).to(torch.float32)
    parameters_nn[:, 1:] = np.log10(parameters_nn[:, 1:])
    parameters_nn = (parameters_nn - X_min[None, :]) / (X_max - X_min)[None, :]
    
    #evaluate the emulator
    with torch.inference_mode():
        spectra_nn = emulator(parameters_nn)
    
    # Postprocess the spectra
    spectra = 10**(spectra_nn * (Y_max - Y_min)[None, :] + Y_min[None, :])
    spectra = spectra.detach().numpy() # Convert back to numpy array
    
    # Compute the likelihood
    log_like = -0.5*np.sum(
        (spectra - sample_spectrum[None, :])**2/(sample_spectrum_err[None, :]**2),
        axis=1
    )
    
    return log_like

In [ ]:
sampler = ultranest.ReactiveNestedSampler(
    param_names, 
    likelihood, 
    prior_transform, 
    vectorized=True
)

In [ ]:
sampler.run();

# Let's see how we did

In [ ]:
sampler.print_results()
print('{:<12} | {:^8} | {:^8} | {:^8}'.format('Name', 'Mean', 'Std', 'True'))
for name, mean, std, true in zip(
    param_names, 
    sampler.results['posterior']['mean'], 
    sampler.results['posterior']['stdev'],
    sample_params[0]
):
    print(f'{name:<12} | {mean:^7.2e} | {std:^7.2e} | {true:^7.2e}')
sampler.plot_corner()

# Check that our fit looks good

In [ ]:
from ultranest.plot import PredictionBand

plt.figure(dpi=150)

plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')

band = PredictionBand(const_c/hz_grid) # Helps visualize posterior prediction intervals
plt.errorbar(const_c/hz_grid, sample_spectrum, yerr=sample_spectrum_err, zorder=-10)

samples = sampler.results['samples']

# Preprocess the samples
parameters_nn = torch.from_numpy(samples).to(torch.float32)
parameters_nn[:, 1:] = np.log10(parameters_nn[:, 1:])
parameters_nn = (parameters_nn - X_min[None, :]) / (X_max - X_min)[None, :]

#evaluate the model
with torch.inference_mode():
    spectra_nn = emulator(parameters_nn)

# Postprocess the spectra
spectra_samples = 10**(spectra_nn * (Y_max - Y_min)[None, :] + Y_min[None, :]).detach().numpy()

# Plot the prediction bands
for spectrum in spectra_samples:
    band.add(spectrum)

band.shade(color='k', alpha=0.2, q=0.995/2)
band.shade(color='k', alpha=0.2, q=0.97/2)
band.shade(color='k', alpha=0.2)

band.line(color='r', label='mean', ls='--', lw=1)

plt.legend()
plt.xscale('log');

# Tasks for you

Can you build a model that provides a more accurate fit?

What happens if you try to fit data whose parameters are outside of the range of the training data?

How is performance impacted with the number of training samples?

How does the quality of the fit change with noisier data?



# Bonus: Spectrum Classifier

In [ ]:
@numba.njit
def create_spectrum_2(nu, T, amp1, amp2, amp3, vel1, vel2, vel3): # A simulated spectrum
    
    BB =  2 * const_h * nu**3 / const_c**2 * 1 / (np.exp((nu * const_h) / (const_kb * T)) - 1)

    BB += P_cygni(nu, 3e14, vel1, 1e-1)*amp1
    BB += P_cygni(nu, 4e14, vel2, 1e-1)*amp2
    BB += P_cygni(nu, 5e14, vel3, 1e-1)*amp3
    
    return BB

@numba.njit(parallel=True)
def gen_spectra_multiclass(parameters, spectrum_classes=0): # Generate spectra from a grid of input parameters

    if isinstance(spectrum_classes, int):
        spectrum_classes = np.ones(parameters.shape[0])*spectrum_classes
    spectra = np.empty((len(parameters), len(hz_grid)))
    for i in numba.prange(len(parameters)):
        T, amp1, amp2, amp3, vel1, vel2, vel3 = parameters[i]
        spec_class = spectrum_classes[i]
        if spec_class == 0:
            spectra[i] = create_spectrum(hz_grid, T, amp1, amp2, amp3, vel1, vel2, vel3)
        else:
            spectra[i] = create_spectrum_2(hz_grid, T, amp1, amp2, amp3, vel1, vel2, vel3)
    return spectra, spectrum_classes

In [ ]:
N = 50000 # Number of examples
hz_grid = np.logspace(12, 15, 1000)
temperatures = np.random.random(N) * (10000 - 2000) + 2000 # Random grid of temperatures
amp1s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5) # Random grids of amplitudes
amp2s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5) 
amp3s = 10**(np.random.random(N) * (-4 - -5.5) + -5.5)

vel1s = 10**(np.random.random(N) * (10 - 9) + 9) # Random grid of Velocities
vel2s = 10**(np.random.random(N) * (10 - 9) + 9)
vel3s = 10**(np.random.random(N) * (10 - 9) + 9)

params = np.vstack((temperatures, amp1s, amp2s, amp3s, vel1s, vel2s, vel3s)).T

spectra, classes = gen_spectra_multiclass(params, spectrum_classes=np.random.randint(0, 2, len(params))) # Generate N spectra

X = params
Y = spectra

In [ ]:
plt.figure(dpi=150)
spectra_class_A = spectra[classes==0]
plt.plot(const_c/hz_grid, spectra_class_A[0])
plt.plot(const_c/hz_grid, spectra_class_A[1])
plt.plot(const_c/hz_grid, spectra_class_A[2])

plt.title('Class A')
plt.xscale('log')
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')

plt.figure(dpi=150)
spectra_class_B = spectra[classes==1]
plt.plot(const_c/hz_grid, spectra_class_B[0])
plt.plot(const_c/hz_grid, spectra_class_B[1])
plt.plot(const_c/hz_grid, spectra_class_B[2])

plt.title('Class B')
plt.xscale('log')
plt.xlabel('Wavelength (cm)')
plt.ylabel('Flux')


In [ ]:
# Create the classifer
class Classifier(nn.Module): 

    def __init__(self, n_input, n_classes, n_hidden):

        super().__init__()

        self.n_input = n_input
        self.n_hidden = n_hidden
        self.n_classes = n_classes

        self.input_layer = nn.Sequential(
            nn.Linear(n_input, n_hidden), # Matrix Multiplication
            nn.Sigmoid() # Activation Function
        )
        self.hidden_layer = nn.Sequential(
            nn.Linear(n_hidden, n_hidden),
            nn.Sigmoid(),
        )
        self.output_layer = nn.Sequential(
            nn.Linear(n_hidden, n_classes),
            nn.Softmax(-1) # Sum of outputs is 1, all outputs from 0 to 1
        )

    def forward(self, x): # Function that takes in an input and returns a prediction
        # x is (Batch x n_input)
        
        x = self.input_layer(x) # (Batch x n_input) -> (Batch x n_hidden)
        x = self.hidden_layer(x) # (Batch x n_hidden) -> (Batch x n_hidden)
        x = self.output_layer(x) # (Batch x n_hidden) -> (Batch x n_classes)
        
        return x

In [ ]:
# Preprocess
X_nn = torch.from_numpy(spectra)
X_nn = torch.log10(X_nn+1e-7).to(torch.float32) # The spectra themselves tend to be log-distributed

X_min = X_nn.min(axis=0).values
X_max = X_nn.max(axis=0).values

X_nn = (X_nn - X_min) / (X_max - X_min) # Scale the spectra from 0 to 1

Y_nn = nn.functional.one_hot(torch.from_numpy(classes)).to(torch.float32) # Class probabilities

In [ ]:
# Split
(X_train, 
 X_val,
 Y_train,
 Y_val) = train_test_split(X_nn, Y_nn, train_size=0.7, random_state=42, shuffle=True)

In [ ]:
classifier = Classifier(
    n_input=X_nn.shape[1], 
    n_classes=2, # 2 classes
    n_hidden=64
)

In [ ]:
learning_rate = 0.0001

optimizer = torch.optim.Adam(
    classifier.parameters(), 
    lr=learning_rate
)

In [ ]:
loss_function = nn.CrossEntropyLoss() # Loss between probabilities of two distributions

In [ ]:
train_model_with_loss_tracking(
    classifier, 
    optimizer, 
    loss_function,
    X_train, 
    Y_train, 
    X_val, 
    Y_val, 
    batch_size=100,
    num_epochs=40,
    show_prediction_plots=False
)

In [ ]:
predictions = classifier(X_val).max(dim=1).indices
truths = Y_val.max(dim=1).indices
accuracy = (predictions == truths).sum()/len(truths)
print("Accuracy:", accuracy.item())

# Tasks for you

Can you build a classifier model that correctly classifies the "observed" spectrum above as class A?

### Things to consider:

How do you handle potential zero or negative values from random noise?

How can this model be modified handle noisy data?